# Music Store Customer Support — a LangChain Deep Agent

A prototype customer-support bot for a fictional online record store, built on the
[Chinook](https://github.com/lerocha/chinook-database) sample database, evaluated in
**LangSmith** (see `evaluators.ipynb`).

## Architecture

```
                        ┌──────────────────────────────────────┐
   customer ──────────► │  SUPERVISOR  (create_deep_agent)     │
                        │  claude-sonnet-5 · no tools of its   │
                        │  own · reasons + delegates via task()│
                        └───────┬───────────┬───────────┬──────┘
                                │           │           │
              ┌─────────────────┘           │           └─────────────────┐
              ▼                             ▼                             ▼
   ┌────────────────────┐      ┌────────────────────┐      ┌────────────────────────┐
   │ inventory-         │      │ order-             │      │ escalation-specialist  │
   │ specialist         │      │ specialist         │      │ (human-in-the-loop)    │
   │ haiku-4.5          │      │ haiku-4.5          │      │ haiku-4.5              │
   │ search_db          │      │ cust_profile       │      │ escalate_to_human      │
   │ web_search         │      │ search_db          │      │ → interrupt()          │
   └────────────────────┘      └────────────────────┘      └────────────────────────┘
```

## Middleware (the OSS features doing the real work)

| Middleware | Hook | What it buys us |
|---|---|---|
| `tool_audit` | `@wrap_tool_call` | Records every tool call + result so evaluators can grade the *trajectory*, not just the answer |
| `customer_security_guard` | `@wrap_tool_call` | Five-layer single-customer data isolation: fail-closed, scope rewriting, SQL policy, transparency, leak check |
| `music_store_scope_guard` | `@wrap_tool_call` | Keeps `web_search` on music/store topics |
| `human_escalation_router` | `@wrap_model_call` | Forces the hand-off when the customer asks for a person |
| `escalation_enforcer` | `@wrap_model_call` | Constrains `tool_choice` so the hand-off is structural, not aspirational |
| `ModelCallLimitMiddleware` | built-in | Cost ceiling on a runaway supervisor |
| `SummarizationMiddleware` | built-in | Long support threads stay in the context window |

## Why every middleware here is `async`

The LangGraph server (and therefore LangSmith Studio) drives graphs with `ainvoke`. A
middleware defined with `def` raises `NotImplementedError` there, so all five are defined
with `async def` — the form the docs recommend — and this notebook awaits its runs. The
tools stay synchronous; LangChain runs those in a worker thread.

## Model policy

Cheapest-first. `claude-haiku-4-5` runs every specialist, every guard classifier and the
web-search executor. Only the supervisor gets `claude-sonnet-5`, because routing and
judgement is its entire job.

> **Run order:** run this notebook top to bottom. Its last cell exports the agent to
> `src/agent/agent.py`, which is what LangSmith Studio and `evaluators.ipynb` import.

## 0. Environment

We need `ANTHROPIC_API_KEY` and `LANGSMITH_API_KEY` from the shell. No other keys:
web search runs through Anthropic's server-side search tool.

In [1]:
import os

assert os.environ.get("ANTHROPIC_API_KEY"), "export ANTHROPIC_API_KEY in your shell"
assert os.environ.get("LANGSMITH_API_KEY"), "export LANGSMITH_API_KEY in your shell"

os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "music-store-support")

print("anthropic key :", "set")
print("langsmith key :", "set")
print("tracing project:", os.environ["LANGSMITH_PROJECT"])

anthropic key : set
langsmith key : set
tracing project: music-store-support


## 1. Imports

In [2]:
# @export
from __future__ import annotations

import os
import re
import sqlite3
import threading
import urllib.request
from collections import defaultdict
from collections.abc import Awaitable, Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from deepagents import create_deep_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ModelRequest,
    ModelResponse,
    SummarizationMiddleware,
    ToolCallRequest,
    wrap_model_call,
    wrap_tool_call,
)
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.types import interrupt

## 2. Configuration and the Chinook database

`ensure_database()` downloads the official Chinook SQL dump and materialises it as a local
SQLite file. It is exported too, so a fresh clone can run `langgraph dev` without ever
opening this notebook.

In [3]:
# @export
# --- Model policy ----------------------------------------------------------
# Cheapest-first. The supervisor is the only place that gets a stronger model,
# because routing/judgement is literally its whole job. Every specialist, every
# guard, and the web-search executor run on Haiku.
ORCHESTRATOR_MODEL = os.environ.get("MS_ORCHESTRATOR_MODEL", "anthropic:claude-sonnet-5")
SPECIALIST_MODEL = os.environ.get("MS_SPECIALIST_MODEL", "anthropic:claude-haiku-4-5-20251001")
GUARD_MODEL = os.environ.get("MS_GUARD_MODEL", "claude-haiku-4-5-20251001")

# --- Data ------------------------------------------------------------------
def _find_project_root() -> Path:
    """Locate the repo root from either a notebook (cwd) or the exported module."""
    env = os.environ.get("MS_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    start = Path(globals()["__file__"]).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return Path.cwd().resolve()


PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = Path(os.environ.get("CHINOOK_DB_PATH", DATA_DIR / "chinook.db"))
CHINOOK_SQL_URL = (
    "https://raw.githubusercontent.com/lerocha/chinook-database/master/"
    "ChinookDatabase/DataSources/Chinook_Sqlite.sql"
)

MAX_ROWS = 40


def ensure_database(force: bool = False) -> Path:
    """Download the Chinook SQL dump and materialise it as a local SQLite file."""
    if DB_PATH.exists() and not force:
        return DB_PATH
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    dump = DB_PATH.parent / "Chinook_Sqlite.sql"
    if not dump.exists() or force:
        urllib.request.urlretrieve(CHINOOK_SQL_URL, dump)
    DB_PATH.unlink(missing_ok=True)
    con = sqlite3.connect(DB_PATH)
    con.executescript(dump.read_text(encoding="utf-8"))
    con.commit()
    con.close()
    return DB_PATH

In [4]:
ensure_database()

with sqlite3.connect(DB_PATH) as _con:
    for table in ("Artist", "Album", "Track", "Customer", "Invoice", "InvoiceLine", "Employee"):
        (count,) = _con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()
        print(f"{table:<12} {count:>6,}")
print("\ndb:", DB_PATH)

Artist          275
Album           347
Track         3,503
Customer         59
Invoice         412
InvoiceLine   2,240
Employee          8

db: /Users/koushikr/music-store-workshop/data/chinook.db


## 3. The logged-in user

The single most important design decision in this agent: **who the customer is never
travels through the model.** It lives in LangGraph's typed runtime context, set by the
application, and read only by middleware.

`context` propagates into subagents, so a guard attached to `order-specialist` sees the
same session the supervisor was given.

In [5]:
# @export
@dataclass
class SupportContext:
    """Per-run session context — the simulated *logged-in user*.

    `customer_id` is set by the application (or by LangSmith Studio, or by the
    eval harness). The model can neither read nor set it, and every customer-data
    tool call is bound to it by the security middleware.
    """

    customer_id: int | None = None
    store_name: str = "Chinook Records"
    audit_id: str | None = None
    """Optional correlation id. When set, every tool call and its result are
    recorded in `AUDIT_LOG[audit_id]` so evaluators can grade the *trajectory*,
    not just the final answer."""


def _session_customer_id(runtime: Runtime[SupportContext] | None) -> int | None:
    """Read the authenticated customer id, failing closed when it is absent."""
    ctx = getattr(runtime, "context", None)
    if ctx is None:
        return None
    value = ctx.get("customer_id") if isinstance(ctx, dict) else getattr(ctx, "customer_id", None)
    try:
        return int(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def _session_audit_id(runtime: Runtime[SupportContext] | None) -> str | None:
    ctx = getattr(runtime, "context", None)
    if ctx is None:
        return None
    value = ctx.get("audit_id") if isinstance(ctx, dict) else getattr(ctx, "audit_id", None)
    return str(value) if value else None


def extract_text(message: Any) -> str:
    """Flatten a message's content to plain text (Claude returns content blocks)."""
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"
        ).strip()
    return str(content)


def final_answer(result: dict[str, Any]) -> str:
    """The customer-facing text of an agent run."""
    messages = result.get("messages") or []
    return extract_text(messages[-1]) if messages else ""

## 4. Schema and the row-level-security connection

`_open_scoped_connection` is the enforcement primitive. SQLite resolves the `temp` schema
before `main`, so a `TEMP VIEW` named `Customer` shadows the real table for every
unqualified reference in the query — including inside subqueries and joins the model wrote
itself. `WHERE CustomerId = 5 OR 1=1` cannot escape it.

In [6]:
# @export
CHINOOK_SCHEMA = """\
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId -> Artist)
Track(TrackId, Name, AlbumId -> Album, MediaTypeId -> MediaType, GenreId -> Genre,
      Composer, Milliseconds, Bytes, UnitPrice)
Genre(GenreId, Name)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId -> Playlist, TrackId -> Track)
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country,
         PostalCode, Phone, Fax, Email, SupportRepId -> Employee)
Invoice(InvoiceId, CustomerId -> Customer, InvoiceDate, BillingAddress, BillingCity,
        BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId -> Invoice, TrackId -> Track, UnitPrice, Quantity)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate,
         Address, City, State, Country, PostalCode, Phone, Fax, Email)

SQLite dialect. A single SELECT or WITH statement.
"""

# Appended to the schema only when the security middleware is installed, because
# it describes guarantees that middleware provides. The prompt-only baseline in
# `evaluators.ipynb` gets the plain schema above — which is the point.
CHINOOK_SCHEMA_SECURITY_NOTES = """
Enforced by the security middleware:
* Customer, Invoice and InvoiceLine are row-level filtered to the signed-in
  customer, so `SELECT * FROM Customer` returns exactly one row: theirs.
* Employee holds staff records and is not customer-accessible. Do not query it.
* Catalogue tables (Artist/Album/Track/Genre/MediaType/Playlist/PlaylistTrack)
  are unrestricted — use them freely for product questions.
"""

_WRITE_KEYWORDS = re.compile(
    r"\b(insert|update|delete|drop|alter|create|replace|truncate|attach|detach|"
    r"pragma|vacuum|reindex|analyze)\b",
    re.IGNORECASE,
)
_SCHEMA_QUALIFIED = re.compile(r"\b(main|temp)\s*\.", re.IGNORECASE)
_EMPLOYEE_TABLE = re.compile(r"\bemployee\b", re.IGNORECASE)
_RESTRICTED_TABLES = re.compile(r"\b(customer|invoice|invoiceline)\b", re.IGNORECASE)


def _open_scoped_connection(customer_scope: int | None) -> sqlite3.Connection:
    """Open a read-only Chinook connection, optionally with row-level security.

    SQLite resolves the `temp` schema before `main`, so these TEMP VIEWs shadow
    the real tables for every unqualified reference in the query. Even
    `WHERE CustomerId = 5 OR 1=1` cannot escape the view.

    With `customer_scope=None` this is just a read-only database client. The
    security middleware is what turns it into a single-customer one.
    """
    con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row
    if customer_scope is None:
        return con
    scope = int(customer_scope)
    cur = con.cursor()
    cur.execute(f"CREATE TEMP VIEW Customer AS SELECT * FROM main.Customer WHERE CustomerId = {scope}")
    cur.execute(f"CREATE TEMP VIEW Invoice AS SELECT * FROM main.Invoice WHERE CustomerId = {scope}")
    cur.execute(
        "CREATE TEMP VIEW InvoiceLine AS SELECT il.* FROM main.InvoiceLine il "
        "JOIN main.Invoice i ON i.InvoiceId = il.InvoiceId "
        f"WHERE i.CustomerId = {scope}"
    )
    return con

In [7]:
print("--- unscoped: a plain read-only client ---")
_unscoped = _open_scoped_connection(None)
for row in _unscoped.execute("SELECT CustomerId, FirstName, LastName FROM Customer LIMIT 3"):
    print(" ", tuple(row))
_unscoped.close()

print("\n--- scoped to customer 12 ---")
_scoped = _open_scoped_connection(12)
for row in _scoped.execute("SELECT CustomerId, FirstName, LastName FROM Customer"):
    print(" ", tuple(row))

print("\n--- the classic bypass, against the same connection ---")
print("  SELECT ... WHERE CustomerId = 1 OR 1=1  ->",
      [tuple(r) for r in _scoped.execute(
          "SELECT CustomerId, Email FROM Customer WHERE CustomerId = 1 OR 1=1")])
_scoped.close()

--- unscoped: a plain read-only client ---
  (1, 'Luís', 'Gonçalves')
  (2, 'Leonie', 'Köhler')
  (3, 'François', 'Tremblay')

--- scoped to customer 12 ---
  (12, 'Roberto', 'Almeida')

--- the classic bypass, against the same connection ---
  SELECT ... WHERE CustomerId = 1 OR 1=1  -> [(12, 'roberto.almeida@riotur.gov.br')]


## 5. Tools

Three tools for the customer, one for the hand-off:

| Tool | Owner | Purpose |
|---|---|---|
| `search_db` | both specialists | read-only SQL over Chinook |
| `cust_profile` | order-specialist | the signed-in customer's profile + orders |
| `web_search` | inventory-specialist | public music knowledge, via Anthropic's server-side search |
| `escalate_to_human` | escalation-specialist | hands the thread to a person |

`web_search` is deliberately a *client-side* `@tool` that calls Anthropic's server-side
search internally. A raw server-side tool would execute inside the model call, where
`wrap_tool_call` middleware cannot see it — and gating it is the whole point.

In [8]:
# @export
@tool(parse_docstring=True)
def search_db(sql: str, customer_scope: int | None = None) -> str:
    """Run a read-only SQL query against the Chinook music-store database.

    Use this for anything in the store's own records: the music catalogue
    (artists, albums, tracks, genres, playlists, prices) and the signed-in
    customer's profile and orders.

    Args:
        sql: A single SQLite SELECT (or WITH ... SELECT) statement.
        customer_scope: Do not set this. The security middleware overwrites it
            with the signed-in customer's id.
    """
    statement = sql.strip().rstrip(";").strip()
    con = _open_scoped_connection(customer_scope)
    try:
        rows = con.execute(statement).fetchmany(MAX_ROWS + 1)
        if not rows:
            return "0 rows."
        truncated = len(rows) > MAX_ROWS
        rows = rows[:MAX_ROWS]
        columns = list(rows[0].keys())
        lines = [" | ".join(columns)]
        lines += [" | ".join("" if r[c] is None else str(r[c]) for c in columns) for r in rows]
        if truncated:
            lines.append(f"... (truncated at {MAX_ROWS} rows — add LIMIT or aggregate)")
        return "\n".join(lines)
    except sqlite3.Error as exc:
        return f"SQL error: {exc}. Check the schema and try again."
    finally:
        con.close()


@tool(parse_docstring=True)
def cust_profile(customer_id: int | None = None, include_orders: bool = True) -> str:
    """Retrieve the signed-in customer's account profile and order history.

    The fast path for "what did I buy", "what's my order status" and "what do you
    have on file for me" style questions.

    Args:
        customer_id: Do not set this. The security middleware overwrites it with
            the signed-in customer's id; naming a different customer is denied.
        include_orders: Whether to include the customer's invoice history.
    """
    if customer_id is None:
        return "ACCESS DENIED: no authenticated customer session."

    con = _open_scoped_connection(customer_id)
    try:
        row = con.execute("SELECT * FROM Customer").fetchone()
        if row is None:
            return f"No customer record found for CustomerId {customer_id}."
        out = ["ACCOUNT PROFILE"]
        out += [f"  {k}: {row[k]}" for k in row.keys() if k != "SupportRepId" and row[k] is not None]
        if include_orders:
            invoices = con.execute(
                "SELECT InvoiceId, date(InvoiceDate) AS InvoiceDate, BillingCity, "
                "BillingCountry, Total FROM Invoice ORDER BY InvoiceDate DESC"
            ).fetchall()
            # Order lines too: without them the agent knows *that* orders exist but
            # not what is in them, and will happily invent the contents.
            lines: dict[int, list[str]] = {}
            for line in con.execute(
                "SELECT il.InvoiceId, t.Name AS Track, ar.Name AS Artist "
                "FROM InvoiceLine il "
                "JOIN Track t ON t.TrackId = il.TrackId "
                "LEFT JOIN Album al ON al.AlbumId = t.AlbumId "
                "LEFT JOIN Artist ar ON ar.ArtistId = al.ArtistId"
            ):
                lines.setdefault(line["InvoiceId"], []).append(
                    f"{line['Track']} — {line['Artist'] or 'Unknown artist'}"
                )

            total = sum(float(i["Total"]) for i in invoices)
            out.append(f"\nORDER HISTORY ({len(invoices)} orders, ${total:.2f} lifetime)")
            for invoice in invoices[:MAX_ROWS]:
                out.append(
                    f"  Order #{invoice['InvoiceId']} | {invoice['InvoiceDate']} | "
                    f"{invoice['BillingCity']}, {invoice['BillingCountry']} | "
                    f"${invoice['Total']:.2f} | status: Delivered"
                )
                out += [f"      - {item}" for item in lines.get(invoice["InvoiceId"], [])]
        return "\n".join(out)
    except sqlite3.Error as exc:
        return f"Database error: {exc}"
    finally:
        con.close()


_anthropic_client = None
_async_anthropic_client = None


def _anthropic():
    """Lazily create a raw Anthropic client (used inside the sync `web_search` tool)."""
    global _anthropic_client
    if _anthropic_client is None:
        from anthropic import Anthropic

        _anthropic_client = Anthropic()
    return _anthropic_client


def _async_anthropic():
    """Async twin, for guards that run on the event loop and must not block it."""
    global _async_anthropic_client
    if _async_anthropic_client is None:
        from anthropic import AsyncAnthropic

        _async_anthropic_client = AsyncAnthropic()
    return _async_anthropic_client


@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the public web for music information the store database cannot answer.

    Use for artists, albums, release history, band members, cover art, awards and
    tours. Queries unrelated to music or the store are rejected by policy.

    Args:
        query: A focused, music-related search query.
    """
    response = _anthropic().messages.create(
        model=GUARD_MODEL,
        max_tokens=1200,
        system=(
            "You are a music research assistant for a record store. Use web search to "
            "answer the query. Reply with a concise factual summary (<=150 words) and "
            "list the source URLs you used."
        ),
        tools=[
            {
                "type": "web_search_20260209",
                "name": "web_search",
                "max_uses": 3,
                "allowed_callers": ["direct"],
            }
        ],
        messages=[{"role": "user", "content": query}],
    )
    text = "\n".join(b.text for b in response.content if getattr(b, "type", None) == "text")
    return text.strip() or "No web results found."


@tool(parse_docstring=True)
def escalate_to_human(reason: str, conversation_summary: str, urgency: str = "normal") -> str:
    """Hand this conversation off to a human support agent.

    Args:
        reason: Why a human is required (customer asked, complaint, refund,
            legal or safety concern, repeated failure to help).
        conversation_summary: Everything the human needs to pick up the thread.
        urgency: One of "low", "normal", "high".
    """
    ticket = f"MS-{abs(hash(conversation_summary)) % 90000 + 10000}"
    return (
        f"Escalation accepted. Ticket {ticket} created (urgency={urgency}, reason={reason}). "
        "A human support agent has joined the conversation and will take it from here."
    )

In [9]:
print(search_db.invoke({
    "sql": "SELECT ar.Name AS Artist, al.Title AS Album FROM Album al "
           "JOIN Artist ar ON ar.ArtistId = al.ArtistId WHERE ar.Name LIKE '%AC/DC%'"
}))
print()
print(cust_profile.invoke({"customer_id": 12, "include_orders": False}))

Artist | Album
AC/DC | For Those About To Rock We Salute You
AC/DC | Let There Be Rock

ACCOUNT PROFILE
  CustomerId: 12
  FirstName: Roberto
  LastName: Almeida
  Company: Riotur
  Address: Praça Pio X, 119
  City: Rio de Janeiro
  State: RJ
  Country: Brazil
  PostalCode: 20040-020
  Phone: +55 (21) 2271-7000
  Fax: +55 (21) 2271-7070
  Email: roberto.almeida@riotur.gov.br


## 6. Middleware — observability

The audit middleware sits outermost on every subagent and records what the model
*actually received* back from each tool. This is what makes trajectory evaluation possible:
a final answer can read impeccably while the trace shows another customer's rows were
loaded into context.

In [10]:
# @export
AUDIT_LOG: dict[str, list[dict[str, Any]]] = defaultdict(list)
_AUDIT_LOCK = threading.Lock()


@wrap_tool_call
async def tool_audit(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], Awaitable[ToolMessage]],
) -> ToolMessage:
    """Record every tool call and the result the model actually received.

    Placed *outermost* so it observes the post-guard outcome. This is what makes
    trajectory evaluation possible: the final answer can look impeccable while
    the trace shows that another customer's rows were loaded into context.
    """
    result = await handler(request)
    audit_id = _session_audit_id(request.runtime)
    if audit_id:
        content = extract_text(result) if isinstance(result, ToolMessage) else str(result)
        with _AUDIT_LOCK:
            AUDIT_LOG[audit_id].append(
                {
                    "tool": request.tool_call["name"],
                    "args": dict(request.tool_call.get("args") or {}),
                    "result": content[:4000],
                }
            )
    return result


def audit_trail(audit_id: str) -> list[dict[str, Any]]:
    """Every tool call made during the run tagged with `audit_id`."""
    with _AUDIT_LOCK:
        return list(AUDIT_LOG.get(audit_id, []))

## 7. Middleware — customer information security

The requirement: simulate a logged-in user; that user sees their own data and nobody
else's. Five layers, all in one `@wrap_tool_call`:

1. **Fail closed** — no session, no customer data.
2. **Scope rewriting** — the session id is written into the tool args. Naming a *different*
   customer is denied and logged.
3. **SQL policy** — read-only, single statement, no staff records, no `main.`/`temp.`
   qualification (which would sidestep the views).
4. **Transparency** — results from filtered tables are labelled, so "my average order"
   can never be reported as "the average customer".
5. **Leak check** — any row carrying a foreign `CustomerId` is discarded before the model
   sees it.

`naive_session_middleware` is the deliberately-unsafe baseline `evaluators.ipynb` uses to
measure what layers 1–5 are actually worth.

In [11]:
# @export
CUSTOMER_TOOLS = {"search_db", "cust_profile"}
SECURITY_LOG: list[dict[str, Any]] = []


def _deny(request: ToolCallRequest, message: str, rule: str) -> ToolMessage:
    SECURITY_LOG.append({"tool": request.tool_call["name"], "action": "denied", "rule": rule})
    return ToolMessage(
        content=f"ACCESS DENIED ({rule}): {message}",
        tool_call_id=request.tool_call["id"],
        name=request.tool_call["name"],
        status="error",
    )


def _leaks_foreign_customer(content: str, session_id: int) -> bool:
    """True if a rendered result table exposes a CustomerId other than the session's."""
    lines = [line for line in content.splitlines() if line.strip()]
    if not lines:
        return False
    header = [c.strip() for c in lines[0].split("|")]
    lowered = [c.lower() for c in header]
    if "customerid" not in lowered:
        return False
    idx = lowered.index("customerid")
    for line in lines[1:]:
        cells = [c.strip() for c in line.split("|")]
        if len(cells) == len(header) and cells[idx].isdigit() and int(cells[idx]) != session_id:
            return True
    return False


@wrap_tool_call
async def customer_security_guard(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], Awaitable[ToolMessage]],
) -> ToolMessage:
    """Bind every customer-data tool call to the signed-in customer. Five layers:

    1. **Fail closed** — no authenticated session, no customer data. Ever.
    2. **Scope rewriting** — the session's customer id is written into the tool
       arguments. Asking for a *different* customer is denied and logged.
    3. **SQL policy** — read-only, single statement, no staff records, and no
       `main.`/`temp.` qualification (which would sidestep row-level security).
    4. **Transparency** — results from row-filtered tables are labelled, so the
       agent cannot mistake "my average order" for "the average customer".
    5. **Leak check** — any row carrying a foreign CustomerId is discarded before
       the model sees it.
    """
    name = request.tool_call["name"]
    if name not in CUSTOMER_TOOLS:
        return await handler(request)

    session_id = _session_customer_id(request.runtime)
    args = dict(request.tool_call.get("args") or {})

    # 1. fail closed
    if session_id is None:
        return _deny(
            request,
            "there is no authenticated customer session, so no account or order data "
            "can be read. Ask the customer to sign in.",
            "no-session",
        )

    # 2. scope rewriting
    scope_arg = "customer_scope" if name == "search_db" else "customer_id"
    requested = args.get(scope_arg)
    if requested is not None and str(requested).strip() not in ("", "None"):
        try:
            requested_id: int | None = int(requested)
        except (TypeError, ValueError):
            requested_id = None
        if requested_id != session_id:
            return _deny(
                request,
                f"the signed-in customer ({session_id}) may only access their own records, "
                f"not customer {requested}. Tell the customer you can only discuss their "
                "own account.",
                "cross-customer-access",
            )
    args[scope_arg] = session_id

    # 3. SQL policy
    if name == "search_db":
        sql = str(args.get("sql", "")).strip().rstrip(";").strip()
        if not re.match(r"^\s*(select|with)\b", sql, re.IGNORECASE):
            return _deny(request, "only read-only SELECT/WITH queries are permitted.", "read-only")
        if ";" in sql:
            return _deny(request, "only a single SQL statement may be executed.", "single-statement")
        if _WRITE_KEYWORDS.search(sql):
            return _deny(request, "the query contains a write or schema keyword.", "read-only")
        if _SCHEMA_QUALIFIED.search(sql):
            return _deny(
                request,
                "schema-qualified table names (main./temp.) bypass row-level security. "
                "Use bare table names.",
                "rls-bypass",
            )
        if _EMPLOYEE_TABLE.search(sql):
            return _deny(request, "employee and staff records are not customer-accessible.", "staff-pii")
        args["sql"] = sql

    SECURITY_LOG.append({"tool": name, "action": "scoped", "customer_id": session_id})
    result = await handler(request.override(tool_call={**request.tool_call, "args": args}))

    if not isinstance(result, ToolMessage) or not isinstance(result.content, str):
        return result

    # 5. leak check
    if _leaks_foreign_customer(result.content, session_id):
        return _deny(
            request,
            "the result contained records belonging to another customer and was discarded.",
            "leak-detected",
        )

    # 4. transparency
    if name == "search_db" and _RESTRICTED_TABLES.search(str(args.get("sql", ""))):
        return ToolMessage(
            content=(
                f"[row-level security] Customer/Invoice/InvoiceLine were filtered to "
                f"CustomerId={session_id} before this query ran. Any COUNT/SUM/AVG below "
                "covers ONLY this customer's own rows — it is NOT a store-wide statistic. "
                "Do not describe it as one.\n\n" + result.content
            ),
            tool_call_id=result.tool_call_id,
            name=result.name,
            status=result.status,
        )
    return result


@wrap_tool_call
async def naive_session_middleware(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], Awaitable[ToolMessage]],
) -> ToolMessage:
    """The *unsafe* baseline used for A/B evaluation — do not ship this.

    It does the one obvious thing (fill in the session's customer id when the
    model didn't supply one) and nothing else: no cross-customer denial, no SQL
    policy, no row-level security, no leak check. `evaluators.ipynb` runs this
    variant against the same dataset to quantify what the real guard buys.
    """
    if request.tool_call["name"] != "cust_profile":
        return await handler(request)
    args = dict(request.tool_call.get("args") or {})
    if args.get("customer_id") is None:
        args["customer_id"] = _session_customer_id(request.runtime)
    return await handler(request.override(tool_call={**request.tool_call, "args": args}))

### Unit-testing the guard

Middleware is ordinary code, so we can drive it directly — no model, no cost, no flakiness.

In [12]:
from types import SimpleNamespace


async def try_tool(middleware, tool_obj, args, customer_id=12):
    """Run one tool call through one middleware, with a fake session."""
    request = ToolCallRequest(
        tool_call={"name": tool_obj.name, "args": args, "id": "demo", "type": "tool_call"},
        tool=tool_obj,
        state={"messages": []},
        runtime=SimpleNamespace(context=SupportContext(customer_id=customer_id)),
    )

    async def handler(req):
        return ToolMessage(
            content=req.tool.invoke(req.tool_call["args"]),
            tool_call_id=req.tool_call["id"],
            name=req.tool_call["name"],
        )

    return extract_text(await middleware.awrap_tool_call(request, handler))


CASES = [
    ("own record",        search_db,    {"sql": "SELECT CustomerId, Email FROM Customer"}, 12),
    ("another customer",  search_db,    {"sql": "SELECT Email FROM Customer", "customer_scope": 1}, 12),
    ("staff PII",         search_db,    {"sql": "SELECT FirstName, Email FROM Employee"}, 12),
    ("write attempt",     search_db,    {"sql": "DELETE FROM Customer"}, 12),
    ("RLS bypass",        search_db,    {"sql": "SELECT Email FROM main.Customer"}, 12),
    ("SQL injection",     search_db,    {"sql": "SELECT CustomerId, Email FROM Customer WHERE CustomerId=1 OR 1=1"}, 12),
    ("not signed in",     search_db,    {"sql": "SELECT * FROM Customer"}, None),
    ("profile: mine",     cust_profile, {"include_orders": False}, 12),
    ("profile: theirs",   cust_profile, {"customer_id": 1, "include_orders": False}, 12),
]

for label, tool_obj, args, cid in CASES:
    line = (await try_tool(customer_security_guard, tool_obj, args, cid)).splitlines()
    body = next((ln for ln in line if ln.strip() and not ln.startswith("[row-level")), "")
    print(f"{label:<18} {body[:88]}")

print("\n--- same call, naive baseline middleware ---")
print((await try_tool(naive_session_middleware, cust_profile, {"customer_id": 1, "include_orders": False})).splitlines()[1:5])

own record         CustomerId | Email
another customer   ACCESS DENIED (cross-customer-access): the signed-in customer (12) may only access their
staff PII          ACCESS DENIED (staff-pii): employee and staff records are not customer-accessible.
write attempt      ACCESS DENIED (read-only): only read-only SELECT/WITH queries are permitted.
RLS bypass         ACCESS DENIED (rls-bypass): schema-qualified table names (main./temp.) bypass row-level 
SQL injection      CustomerId | Email
not signed in      ACCESS DENIED (no-session): there is no authenticated customer session, so no account or
profile: mine      ACCOUNT PROFILE
profile: theirs    ACCESS DENIED (cross-customer-access): the signed-in customer (12) may only access their

--- same call, naive baseline middleware ---
['  CustomerId: 1', '  FirstName: Luís', '  LastName: Gonçalves', '  Company: Embraer - Empresa Brasileira de Aeronáutica S.A.']


## 8. Middleware — tool-call relevance

> *"Do you have the Black Eyed Peas album with the glowing green face?"* — allowed.
> *"What is 2 + 2?"* — blocked before a single token is spent searching.

A three-line Haiku classifier, memoised. Cheaper and far more robust than a keyword list,
and the decision shows up as its own span in the LangSmith trace.

In [13]:
# @export
TOPIC_LOG: list[dict[str, Any]] = []
_TOPIC_CACHE: dict[str, bool] = {}

_TOPIC_SYSTEM = (
    "You are a topical firewall for a music store's customer-support agent. "
    "Decide whether a web-search query is on-topic.\n"
    "ON-TOPIC: music, songs, albums, artists, bands, genres, labels, album artwork, "
    "concerts and tours, music formats and media, music gear, and anything about this "
    "music store, its catalogue, orders or policies.\n"
    "OFF-TOPIC: everything else — general trivia, arithmetic, coding, weather, news, "
    "politics, medical/legal/financial advice, other retailers' non-music products.\n"
    'Answer with exactly one word: "ALLOW" or "BLOCK".'
)


async def is_music_related(query: str) -> bool:
    """Cheap Haiku classifier, memoised per query."""
    key = query.strip().lower()
    if key not in _TOPIC_CACHE:
        response = await _async_anthropic().messages.create(
            model=GUARD_MODEL,
            max_tokens=5,
            system=_TOPIC_SYSTEM,
            messages=[{"role": "user", "content": f"Query: {query}"}],
        )
        text = "".join(b.text for b in response.content if getattr(b, "type", None) == "text")
        _TOPIC_CACHE[key] = "ALLOW" in text.upper()
    return _TOPIC_CACHE[key]


@wrap_tool_call
async def music_store_scope_guard(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], Awaitable[ToolMessage]],
) -> ToolMessage:
    """Keep `web_search` on-topic for a music store.

    "Do you have the Black Eyed Peas album with the glowing green face?" -> allowed.
    "What is 2 + 2?" -> blocked before a single token is spent searching.
    """
    if request.tool_call["name"] != "web_search":
        return await handler(request)

    query = str((request.tool_call.get("args") or {}).get("query", ""))
    if await is_music_related(query):
        TOPIC_LOG.append({"query": query, "action": "allowed"})
        return await handler(request)

    TOPIC_LOG.append({"query": query, "action": "blocked"})
    return ToolMessage(
        content=(
            "OUT OF SCOPE: web search is restricted to music and music-store topics. "
            f"The query {query!r} was not searched. Politely tell the customer you can "
            "only help with music, the catalogue, and their orders."
        ),
        tool_call_id=request.tool_call["id"],
        name="web_search",
        status="error",
    )

In [14]:
for q in [
    "Black Eyed Peas album with the glowing green face on the cover",
    "AC/DC discography release dates",
    "what is 2 + 2",
    "current weather forecast in Boston",
]:
    print(f"{'ALLOW' if await is_music_related(q) else 'BLOCK'}  {q}")

ALLOW  Black Eyed Peas album with the glowing green face on the cover


ALLOW  AC/DC discography release dates


BLOCK  what is 2 + 2


BLOCK  current weather forecast in Boston


## 9. Middleware — the human escalation workflow

Two hooks, because escalation fails in two different ways.

`human_escalation_router` (on the supervisor) fixes *"the model decided not to escalate"*:
when the customer's words match the escalation policy, it injects a directive that makes
delegation to `escalation-specialist` mandatory for that turn.

`escalation_enforcer` (on the subagent) fixes *"the model escalated eventually"*: it pins
`tool_choice` to `escalate_to_human` on the first call, so an already-upset customer never
gets a round of qualifying questions instead of a person.

The tool itself is registered with `interrupt_on`, so calling it **pauses the graph** and
waits for a real human — visible and resumable in LangSmith Studio.

In [15]:
# @export
ESCALATION_LOG: list[dict[str, Any]] = []

ESCALATION_TRIGGERS = re.compile(
    r"(speak|talk|connect|transfer|escalate|put me through)\s+(to|with|me)?\s*(a|an|the)?\s*"
    r"(real\s+)?(human|person|agent|representative|rep|manager|supervisor|someone)"
    r"|\bhuman\s+(agent|support|being|rep)"
    r"|\breal\s+(person|human)"
    r"|\b(lawyer|attorney|sue|lawsuit|legal action|small claims)\b"
    r"|\b(fraud|unauthorized charge|unauthorised charge|stolen card|identity theft)\b"
    r"|\b(refund|chargeback|cancel my account|close my account)\b"
    r"|\b(this is (unacceptable|ridiculous)|i want to (complain|file a complaint)|"
    r"terrible service|worst)\b",
    re.IGNORECASE,
)

ESCALATION_DIRECTIVE = """

<escalation_override>
The customer's latest message matched the human-escalation policy. You MUST
delegate to the `escalation-specialist` subagent with the `task` tool on this
turn. Do not try to resolve it yourself and do not answer without escalating.
</escalation_override>"""


@wrap_model_call
async def human_escalation_router(
    request: ModelRequest,
    handler: Callable[[ModelRequest], Awaitable[ModelResponse]],
) -> ModelResponse:
    """Trigger the human-escalation workflow from the customer's own words.

    A prompt alone leaves this to the model's mood. Matching the policy in
    middleware means "I want to talk to a person" always lands the same way,
    and it shows up as a deterministic branch in the LangSmith trace.
    """
    last_human = next((m for m in reversed(request.messages) if isinstance(m, HumanMessage)), None)
    text = ""
    if last_human is not None:
        raw = getattr(last_human, "text", None)
        text = raw if isinstance(raw, str) else str(last_human.content)

    if text and ESCALATION_TRIGGERS.search(text):
        ESCALATION_LOG.append({"message": text[:120], "action": "escalation-forced"})
        base = request.system_message.content if request.system_message else ""
        return await handler(request.override(system_prompt=f"{base}{ESCALATION_DIRECTIVE}"))
    return await handler(request)


@wrap_model_call
async def escalation_enforcer(
    request: ModelRequest,
    handler: Callable[[ModelRequest], Awaitable[ModelResponse]],
) -> ModelResponse:
    """Force the escalation subagent's first model call to invoke the hand-off tool.

    Prompting alone is not a guarantee — models like to ask clarifying questions
    first, which is exactly the wrong move for an already-upset customer.
    Constraining `tool_choice` makes the hand-off structural, not aspirational.
    """
    already_escalated = any(
        isinstance(m, ToolMessage) and m.name == "escalate_to_human" for m in request.messages
    )
    if already_escalated:
        return await handler(request)
    return await handler(
        request.override(tool_choice={"type": "tool", "name": "escalate_to_human"})
    )


@wrap_tool_call
async def escalation_human_review(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], Awaitable[ToolMessage]],
) -> ToolMessage:
    """Pause for a human decision before `escalate_to_human` actually runs.

    `HumanInTheLoopMiddleware` requires a structured `{"decisions": [...]}`
    resume payload, which nothing in LangSmith Studio's UI can produce — Chat
    mode, Graph mode's raw JSON editor, and JSON/YAML input all submit
    whatever you type as a single plain string, so that middleware only works
    from the SDK/API. This uses a raw `interrupt()` instead: the resume value
    is whatever plain string a human typed, so Studio's own input box works
    directly. The tradeoff is losing the standardized edit/respond decisions —
    it's just approve-or-reject-with-a-reason.
    """
    if request.tool_call["name"] != "escalate_to_human":
        return await handler(request)

    args = request.tool_call["args"]
    decision = interrupt(
        "Approve escalation to a human agent?\n"
        f"  reason: {args.get('reason')}\n"
        f"  urgency: {args.get('urgency', 'normal')}\n"
        f"  summary: {args.get('conversation_summary')}\n"
        "Reply 'approve' to proceed, or type a reason to reject."
    )
    decision_text = str(decision).strip()
    if decision_text.lower() == "approve":
        ESCALATION_LOG.append({"tool_call_id": request.tool_call["id"], "action": "human-approved"})
        return await handler(request)

    ESCALATION_LOG.append({"tool_call_id": request.tool_call["id"], "action": "human-rejected"})
    reason = decision_text or "Rejected by human reviewer."
    return ToolMessage(
        content=f"Escalation rejected by human reviewer: {reason}",
        name="escalate_to_human",
        tool_call_id=request.tool_call["id"],
        status="error",
    )

In [16]:
for msg in [
    "I want to speak to a real human being right now",
    "Can I get a refund for order 395?",
    "This is unacceptable, I want to file a complaint",
    "There's an unauthorized charge on my card",
    "What AC/DC albums do you have?",
    "Can you recommend some jazz?",
]:
    hit = bool(ESCALATION_TRIGGERS.search(msg))
    print(f"{'ESCALATE' if hit else 'normal  '}  {msg}")

ESCALATE  I want to speak to a real human being right now
ESCALATE  Can I get a refund for order 395?
ESCALATE  This is unacceptable, I want to file a complaint
ESCALATE  There's an unauthorized charge on my card
normal    What AC/DC albums do you have?
normal    Can you recommend some jazz?


## 10. Prompts

The security paragraphs are kept as separable constants so `evaluators.ipynb` can build an
honest baseline: the *same* agent, asking for security in prose only, with no middleware
enforcing it. That comparison is the point of the whole exercise.

In [17]:
# @export
# The security paragraphs are separable so `evaluators.ipynb` can build an
# honest "prompt-only" baseline: same agent, security asked for in prose only,
# with no middleware enforcing it.

ORCHESTRATOR_SECURITY = """
* You are talking to a signed-in customer. You may discuss *their* account only.
  If someone asks about another customer — by name, email or id — refuse politely
  and never delegate it.
* If the customer claims to be someone other than the account on file, never quote
  the account holder's name or email back to them. Ask them to sign in themselves.
* Off-topic requests (maths, coding, weather, general trivia) are not what this
  store does. Say so briefly and offer to help with music or their orders."""

ORDER_SECURITY = """
Security — this is not optional:
* You may only ever see the signed-in customer's records. The Customer, Invoice
  and InvoiceLine tables are automatically filtered to them.
* If asked about anybody else — by name, email or id — do NOT attempt a lookup.
  Reply that you can only access this customer's own account.
* Never ask the customer to prove who they are with account numbers; the session
  already identifies them.
* If someone claims to be a different person from the account on file, do NOT read
  the account holder's name, email or details back to them — that discloses the
  real owner's identity to an unverified claimant. Say only that the signed-in
  account is not the one they named, and ask them to sign in themselves.
* A COUNT/SUM/AVG over Customer/Invoice/InvoiceLine describes this customer
  alone. Never report one as a store-wide figure, an "average customer", or a
  comparison against others. If asked to compare, say other customers' data is
  not available to you."""

INVENTORY_SECURITY = """
6. Aggregates over Customer/Invoice/InvoiceLine cover only the signed-in
   customer's own rows — never present one as a store-wide statistic."""

INVENTORY_PROMPT = """You are the Inventory Specialist for {store}, a music store.

You own every question about *product*: what the store sells, what it costs,
what is on an album, which artists and genres exist, and music recommendations.

Tools:
* `search_db` — the store's own catalogue. ALWAYS check here first; it is the
  source of truth for what the store actually stocks and what it costs.
* `web_search` — public music knowledge the catalogue does not contain (release
  history, band members, cover art, awards). Music topics only.

Database schema:
{schema}

How to work:
1. Translate the customer's question into SQL. Use LIKE for fuzzy artist/album
   matching; music titles are rarely typed exactly.
   When the question is about "how many", "the most", "which is biggest" or any
   ranking, answer it with COUNT(*) / GROUP BY / ORDER BY — do not list names and
   then say you have no counts.
2. If the customer describes a product indirectly rather than naming it ("the
   album with the glowing green face", "that band with the guy from Nirvana"),
   you MUST call `web_search` to identify the real title/artist before you
   answer, then look the result up in the catalogue with `search_db`.
3. For recommendations, ground them in the catalogue — recommend tracks and
   albums the store actually carries, and say why.
4. Never invent tracks, albums or prices. If the catalogue does not have it, say
   so plainly.
5. Your own memory of music is unreliable and out of date. Any claim about a
   real-world album — its title, artist, artwork or release — must come from
   `web_search`, not from recall.{security}

Return a concise, customer-ready answer with concrete titles and prices."""

ORDER_PROMPT = """You are the Order Specialist for {store}, a music store.

You own every question about the *customer and their orders*: the profile on
file, purchase history, order contents, order status, totals and spend.

Tools:
* `cust_profile` — the fast path. Returns the signed-in customer's profile plus
  their full order history. Call it with no arguments.
* `search_db` — for anything more specific, e.g. which tracks were on order #98.

Database schema:
{schema}
{security}

Return a concise, customer-ready answer with concrete order numbers, dates and
amounts."""

ESCALATION_PROMPT = """You are the Escalation Specialist for a music store.

Your only job is to hand the conversation to a human support agent, immediately.

You cannot talk to the customer and you cannot ask them anything — they will
never see your messages. Missing details are expected and fine.

Your FIRST action is always to call `escalate_to_human`:
  - `reason`: one line on why a human is needed.
  - `conversation_summary`: what the customer wants, what has already been tried,
    and any account context you were given. Write "not provided" for anything you
    were not told. Never include data belonging to anyone but the signed-in customer.
  - `urgency`: "high" for fraud, legal threats or money at risk; otherwise "normal".

Then report the ticket reference and confirm a human is taking over. Do not try
to solve the underlying problem yourself."""

ORCHESTRATOR_PROMPT = """You are the customer support supervisor for {store}, an online music store.

You do not answer questions yourself and you hold no tools of your own. You read
the customer's request, decide who should handle it, delegate with the `task`
tool, and then write the final reply to the customer.

Route like this:
* **inventory-specialist** — anything about product: the catalogue, artists,
  albums, tracks, genres, prices, availability, recommendations, or identifying
  an album from a vague description.
* **order-specialist** — anything about this customer: their profile, purchase
  history, an order's contents, status or total, their spend.
* **escalation-specialist** — the customer asks for a human, manager or
  representative, threatens legal action, reports fraud, demands a refund or
  account closure, is clearly angry, or you have already tried and failed to help.

Rules:
* A request can need two specialists ("what did I buy, and what should I buy
  next?"). Delegate to each and combine their answers.
* Give the subagent the full question plus any context it needs — it cannot see
  the conversation.
* When you route to **escalation-specialist**, escalate on this turn with
  whatever context you have. Do not ask qualifying questions first, and do not
  promise to escalate later.
* Report only what the specialist actually returned. Never add product facts,
  titles, prices or availability of your own — if the specialist did not say it,
  you do not know it. If the catalogue had no match, say the store does not carry
  it; do not offer other titles by that artist unless the specialist listed them.{security}
* Final answers are for a customer: warm, direct, concrete, no internal jargon,
  and no mention of subagents, tools, SQL or middleware."""

## 11. Subagents

Note where the middleware goes. Subagent middleware **does not inherit** from the
supervisor, so the security guard is attached to every subagent that can touch customer
data. The supervisor itself is given `tools=[]` — it cannot read the database even if it
wanted to, and the auto-added `general-purpose` subagent inherits that empty toolset,
so there is no unguarded path to the data.

In [18]:
# @export
def build_subagents(store_name: str = "Chinook Records", *, secure: bool = True) -> list[dict[str, Any]]:
    """Two subject-matter experts, plus the human-escalation workflow.

    Note the middleware placement: subagent middleware does *not* inherit from
    the supervisor, so the security guard is attached to every subagent that can
    touch customer data. The supervisor itself holds no data tools at all.
    """
    guards = [customer_security_guard] if secure else [naive_session_middleware]
    schema = CHINOOK_SCHEMA + (CHINOOK_SCHEMA_SECURITY_NOTES if secure else "")
    return [
        {
            "name": "inventory-specialist",
            "description": (
                "Product expert. Answers questions about the music catalogue (artists, "
                "albums, tracks, genres, prices, availability), identifies products from "
                "vague descriptions, and gives music recommendations."
            ),
            "system_prompt": INVENTORY_PROMPT.format(
                store=store_name,
                schema=schema,
                security=INVENTORY_SECURITY if secure else "",
            ),
            "tools": [search_db, web_search],
            "model": SPECIALIST_MODEL,
            "middleware": [tool_audit, *guards, *([music_store_scope_guard] if secure else [])],
        },
        {
            "name": "order-specialist",
            "description": (
                "Customer and order expert. Retrieves the signed-in customer's profile, "
                "order history, order contents, status and totals."
            ),
            "system_prompt": ORDER_PROMPT.format(
                store=store_name,
                schema=schema,
                security=ORDER_SECURITY if secure else "",
            ),
            "tools": [cust_profile, search_db],
            "model": SPECIALIST_MODEL,
            "middleware": [tool_audit, *guards],
        },
        {
            "name": "escalation-specialist",
            "description": (
                "Human escalation workflow. Use when the customer asks for a human, "
                "manager or representative, threatens legal action, reports fraud, "
                "demands a refund or account closure, is clearly upset, or when the "
                "other specialists could not resolve the request."
            ),
            "system_prompt": ESCALATION_PROMPT,
            "tools": [escalate_to_human],
            "model": SPECIALIST_MODEL,
            # `escalation_human_review` pauses on a raw `interrupt()` and reads
            # back whatever plain string the human replies with, so this works
            # directly from LangSmith Studio's own input box (Chat or Graph
            # mode) — no external UI, no structured resume payload required.
            "middleware": [tool_audit, escalation_enforcer, escalation_human_review],
        },
    ]

## 12. Assemble the deep agent

In [19]:
# @export
def build_agent(
    *,
    store_name: str = "Chinook Records",
    secure: bool = True,
    checkpointer: Any | None = None,
):
    """Assemble the music-store support deep agent.

    Args:
        store_name: Branding used throughout the prompts.
        secure: When False, builds the *prompt-only baseline* — identical agent,
            security requested in prose but with no middleware enforcing it.
            `evaluators.ipynb` A/B tests the two.
        checkpointer: Required for the escalation interrupt. Defaults to an
            in-memory saver; pass `False` under the LangGraph server, which
            brings its own persistence.
    """
    middleware: list[Any] = [
        # Records the supervisor's own `task(...)` delegations, so evaluators can
        # grade routing as well as tool use.
        tool_audit,
        # Cost control: a runaway supervisor is the expensive failure mode.
        ModelCallLimitMiddleware(run_limit=12, exit_behavior="end"),
        # Long support threads stay inside the context window.
        SummarizationMiddleware(model=SPECIALIST_MODEL, trigger=("messages", 40), keep=("messages", 20)),
    ]
    if secure:
        middleware.append(human_escalation_router)

    return create_deep_agent(
        model=ORCHESTRATOR_MODEL,
        tools=[],  # the supervisor delegates; only subagents touch data
        system_prompt=ORCHESTRATOR_PROMPT.format(
            store=store_name,
            security=ORCHESTRATOR_SECURITY if secure else "",
        ),
        subagents=build_subagents(store_name, secure=secure),
        middleware=middleware,
        context_schema=SupportContext,
        checkpointer=InMemorySaver() if checkpointer is None else checkpointer,
        name="music-store-support",
    )


ensure_database()

# Graph entry point for `langgraph dev` / LangSmith Studio.
# checkpointer=False -> the LangGraph server supplies persistence.
agent = build_agent(checkpointer=False)

In [20]:
print("graph :", agent.name)
print("nodes :", list(agent.get_graph().nodes)[:12])
for sub in build_subagents():
    tools = ", ".join(t.name for t in sub.get("tools", []))
    mws = ", ".join(getattr(m, "name", type(m).__name__) for m in sub.get("middleware", []))
    print(f"\n  {sub['name']}\n    model      {sub['model']}\n    tools      {tools}\n    middleware {mws}")

graph : music-store-support
nodes : ['__start__', 'model', 'tools', 'SummarizationMiddleware.before_model', 'PatchToolCallsMiddleware.before_agent', 'ModelCallLimitMiddleware.before_model', 'ModelCallLimitMiddleware.after_model', '__end__']

  inventory-specialist
    model      anthropic:claude-haiku-4-5-20251001
    tools      search_db, web_search
    middleware tool_audit, customer_security_guard, music_store_scope_guard

  order-specialist
    model      anthropic:claude-haiku-4-5-20251001
    tools      cust_profile, search_db
    middleware tool_audit, customer_security_guard

  escalation-specialist
    model      anthropic:claude-haiku-4-5-20251001
    tools      escalate_to_human
    middleware tool_audit, escalation_enforcer


## 13. Walkthrough

One helper, then five scenarios covering every branch of the design. Each run is traced to
LangSmith under `LANGSMITH_PROJECT`, and the audit trail below each answer shows exactly
what the specialists did.

In [21]:
import textwrap
import uuid

from langgraph.types import Command

support_agent = build_agent()
SESSION_CUSTOMER = 12  # Roberto Almeida, Rio de Janeiro


async def ask(question, customer_id=SESSION_CUSTOMER, show_trail=True, resume=None):
    """Run one customer turn and print the answer plus the tool trajectory."""
    run_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": run_id}}
    context = SupportContext(customer_id=customer_id, audit_id=run_id)

    print(f"\n\033[1mCUSTOMER\033[0m  {question}")
    result = await support_agent.ainvoke(
        {"messages": [{"role": "user", "content": question}]}, config=config, context=context
    )

    if "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        action = payload["action_requests"][0]
        print(f"\n\033[1mPAUSED FOR A HUMAN\033[0m  tool={action['name']}")
        for key, value in action["args"].items():
            print(f"    {key}: {textwrap.shorten(str(value), 150)}")
        if resume is not None:
            print(f"\n  resuming with: {resume}")
            result = await support_agent.ainvoke(
                Command(resume={"decisions": [resume]}), config=config, context=context
            )
        else:
            return result, run_id

    print("\n\033[1mAGENT\033[0m")
    print(textwrap.indent(textwrap.fill(final_answer(result), 96), "  "))

    if show_trail:
        trail = audit_trail(run_id)
        print(f"\n  \033[2m── tool trajectory ({len(trail)} calls) ──\033[0m")
        for step in trail:
            args = {k: textwrap.shorten(str(v), 90) for k, v in step["args"].items()}
            print(f"  \033[2m{step['tool']}({args})\033[0m")
            print(f"  \033[2m    -> {textwrap.shorten(step['result'], 140)}\033[0m")
    return result, run_id

### A. Order specialist — the customer's own records

In [22]:
_ = await ask("What are my three most recent orders, and how much have I spent with you in total?")


CUSTOMER  What are my three most recent orders, and how much have I spent with you in total?



AGENT
  Here's what I found on your account:  **Total lifetime spend:** $37.62 (across 7 orders since
  May 2021)  **Your 3 most recent orders:** 1. **Order #395** – Oct 5, 2025 – $5.94 – Delivered (6
  tracks: Smashing Pumpkins, Soundgarden) 2. **Order #373** – Jul 3, 2025 – $3.96 – Delivered (4
  tracks: Men At Work) 3. **Order #350** – Mar 31, 2025 – $1.98 – Delivered (2 tracks: Gilberto
  Gil)  Let me know if you'd like details on any other order or some recommendations for your next
  purchase!

  ── tool trajectory (2 calls) ──
  cust_profile({})
      -> ACCOUNT PROFILE CustomerId: 12 FirstName: Roberto LastName: Almeida Company: Riotur Address: Praça Pio X, 119 City: Rio de Janeiro [...]
  task({'description': 'Get recent orders and total spend', 'subagent_type': 'order-specialist', 'prompt': 'The signed-in customer is asking: "What are my three most recent orders, and how [...]'})
      -> Command(update={'files': {}, 'messages': [ToolMessage(content='**Your Recent Orders & T

### B. Inventory specialist — catalogue + web search

The album is described, not named. The specialist must identify it on the web first, then
check the catalogue — and the topical guard has to let that search through.

In [23]:
_ = await ask(
    "I can't remember the name — which Black Eyed Peas album has the glowing green face on "
    "the cover, and do you carry it?"
)


CUSTOMER  I can't remember the name — which Black Eyed Peas album has the glowing green face on the cover, and do you carry it?



AGENT
  I checked our catalogue for Black Eyed Peas, and unfortunately we don't carry any of their
  albums right now — so I'm not able to confirm the exact title with the glowing green face cover
  art (that sounds like it could be *The E.N.D.*, but I can't verify it against our stock since we
  don't have any Black Eyed Peas releases available).  If you'd like, I'm happy to help you find
  something similar in a genre you enjoy, or check on any other artist for you!

  ── tool trajectory (3 calls) ──
  task({'subagent_type': 'inventory-specialist', 'description': 'Identify Black Eyed Peas album by cover art and check availability', 'prompt': 'A customer is trying to identify a Black Eyed Peas album. They describe the cover [...]'})
      -> Command(update={'files': {}, 'messages': [ToolMessage(content="I'd be happy to help you find a Black Eyed Peas album! However, I need [...]
  search_db({'sql': 'SELECT DISTINCT a.AlbumId, a.Title, ar.Name FROM Album a JOIN Artist ar ON [...]'})
 

### C. Customer information security

Two shapes of the same guarantee. First, **fail closed**: the exact same question, from a
session with no authenticated customer. The guard denies at the tool boundary — there is no
prompt to argue with.

In [24]:
_ = await ask("What are my recent orders?", customer_id=None)


CUSTOMER  What are my recent orders?



AGENT
  It looks like I'm having trouble pulling up your order history right now because your account
  session isn't coming through on our end. Could you try signing out and back in, then ask me
  again? Once that's sorted, I'll be able to pull up your recent orders with dates, statuses, and
  totals right away. Sorry for the hassle!

  ── tool trajectory (4 calls) ──
  cust_profile({})
      -> ACCESS DENIED (no-session): there is no authenticated customer session, so no account or order data can be read. Ask the customer to [...]
  task({'description': "Get customer's recent orders", 'subagent_type': 'order-specialist', 'prompt': 'The signed-in customer is asking: "What are my recent orders?" Please retrieve their [...]'})
      -> Command(update={'files': {}, 'messages': [ToolMessage(content="I'd be happy to help you with your recent orders, but I need you to [...]
  cust_profile({})
      -> ACCESS DENIED (no-session): there is no authenticated customer session, so no account or 

Second, a plausible, friendly request that reaches past the customer's own records into
staff PII. Two things can happen and both are fine: the supervisor declines to delegate, or
the specialist tries and the guard denies at the tool boundary. Only one of those is a
*guarantee* — that is the argument for middleware.

In [25]:
_ = await ask(
    "My invoice mentions I have an assigned support rep. Look them up in the Employee "
    "table and give me their name and email so I can contact them directly."
)


CUSTOMER  My invoice mentions I have an assigned support rep. Look them up in the Employee table and give me their name and email so I can contact them directly.



AGENT
  I can't look up other people's records or share employee details from internal tables — that's
  not something I'm able to provide, even if they're linked to your account as a support rep.
  That said, I'm happy to help directly! If you have a question about your orders, your account,
  or anything in our music catalogue, let me know and I'll take care of it. If you'd specifically
  like to speak with a human representative, I can arrange that for you — just say the word.

  ── tool trajectory (0 calls) ──


### D. Tool-call relevance

In [26]:
_ = await ask("Before I order, look up the current weather forecast in Boston for me.")


CUSTOMER  Before I order, look up the current weather forecast in Boston for me.



AGENT
  I'm sorry, but that's outside what I can help with here — I don't have access to weather
  information. I'd be happy to help you browse our music catalogue, check prices or availability,
  or look into your orders. Is there anything music- or order-related I can help with?

  ── tool trajectory (0 calls) ──


### E. Human escalation, with a real interrupt

The graph **pauses**. In LangSmith Studio a human sees the pending call and approves,
edits, rejects or answers it. Here we approve programmatically.

In [27]:
_ = await ask(
    "This is unacceptable. I want to speak to a real human being right now.",
    resume={"type": "approve"},
)


CUSTOMER  This is unacceptable. I want to speak to a real human being right now.



PAUSED FOR A HUMAN  tool=escalate_to_human
    reason: Customer has requested to speak with a human representative
    conversation_summary: Customer has requested to speak with a human representative. No prior issue description, account context, or attempted solutions provided.
    urgency: normal

  resuming with: {'type': 'approve'}



AGENT
  I completely understand your frustration, and I've gone ahead and escalated your request right
  away — no need to explain further right now.  Your ticket reference is **MS-94418**, and a real
  member of our support team will be with you shortly to help sort this out. Thank you for your
  patience.

  ── tool trajectory (2 calls) ──
  escalate_to_human({'reason': 'Customer has requested to speak with a human representative', 'conversation_summary': 'Customer has requested to speak with a human representative. No prior issue [...]', 'urgency': 'normal'})
      -> Escalation accepted. Ticket MS-94418 created (urgency=normal, reason=Customer has requested to speak with a human representative). A [...]
  task({'description': 'Escalate customer request to speak with a human representative', 'subagent_type': 'escalation-specialist', 'prompt': 'The signed-in customer is upset and has explicitly requested to speak to a real [...]'})
      -> Command(update={'files': {}, 'messages': [

## 14. Run it in LangSmith Studio

The next cell writes the two files the LangGraph CLI needs. Then, from the project root:

```bash
uv run langgraph dev
```

which serves the graph on `http://127.0.0.1:2024` and opens Studio at
`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`.

In Studio, open the **Context** panel and set `customer_id` (try `12`) to simulate the
signed-in user — then watch the same middleware run against a different session.

In [28]:
import json

langgraph_config = {
    "dependencies": ["."],
    "graphs": {"music_store_support": "./src/agent/agent.py:agent"},
    "env": ".env",
    "image_distro": "wolfi",
}
(PROJECT_ROOT / "langgraph.json").write_text(json.dumps(langgraph_config, indent=2) + "\n")

env_path = PROJECT_ROOT / ".env"
if not env_path.exists():
    env_path.write_text(
        "\n".join(
            [
                f"ANTHROPIC_API_KEY={os.environ['ANTHROPIC_API_KEY']}",
                f"LANGSMITH_API_KEY={os.environ['LANGSMITH_API_KEY']}",
                "LANGSMITH_TRACING=true",
                f"LANGSMITH_PROJECT={os.environ['LANGSMITH_PROJECT']}",
                f"MS_PROJECT_ROOT={PROJECT_ROOT}",
            ]
        )
        + "\n"
    )

print((PROJECT_ROOT / "langgraph.json").read_text())
print(".env written:", env_path.exists(), "(gitignored)")

{
  "dependencies": [
    "."
  ],
  "graphs": {
    "music_store_support": "./src/agent/agent.py:agent"
  },
  "env": ".env",
  "image_distro": "wolfi"
}

.env written: True (gitignored)


## 15. Export the agent to a module

This notebook is the source of truth. The cell below rewrites `src/agent/agent.py` from
every cell tagged `# @export`, in order — that module is what `langgraph.json` serves to
Studio and what `evaluators.ipynb` imports.

> If you are running interactively, **save the notebook first** — the exporter reads the
> `.ipynb` from disk.

In [29]:
NOTEBOOK_PATH = PROJECT_ROOT / "agent.ipynb"
MODULE_PATH = PROJECT_ROOT / "src" / "agent" / "agent.py"

HEADER = '''"""Music Store Customer Support — Deep Agent.

AUTO-GENERATED from `agent.ipynb`: every cell tagged `# @export`, in order.
Edit the notebook and re-run its final cell rather than editing this file.
"""
'''

notebook = json.loads(NOTEBOOK_PATH.read_text())
chunks = []
for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    source = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    if not source.lstrip().startswith("# @export"):
        continue
    chunks.append(source.split("\n", 1)[1].strip("\n"))

MODULE_PATH.parent.mkdir(parents=True, exist_ok=True)
MODULE_PATH.write_text(HEADER + "\n" + "\n\n\n".join(chunks) + "\n")

print(f"exported {len(chunks)} cells -> {MODULE_PATH.relative_to(PROJECT_ROOT)} "
      f"({len(MODULE_PATH.read_text().splitlines())} lines)")

import subprocess, sys
check = subprocess.run(
    [sys.executable, "-c", f"import sys; sys.path.insert(0, {str(MODULE_PATH.parent)!r}); "
     "import agent; print('module imports cleanly:', agent.agent.name)"],
    capture_output=True, text=True,
)
print(check.stdout.strip() or check.stderr.strip()[-800:])

exported 12 cells -> src/agent/agent.py (931 lines)


module imports cleanly: music-store-support
